<a id="top"></a>
# Gate baseline and inverse QFT benchmark

**Run All** compares X–X and H–H round trips on the four BB84 states and
measures how often Bob's inverse QFT decodes known words at 2 and 4 qubits.

> **Hardware is explicit opt-in.** The committed `RUN_MODE = "ideal"` stays
> on local Aer and never reads credentials or submits a job. Change it to
> `"hardware"` only after reviewing the hardware settings and local
> `apikey.json`.


## Configuration

Edit only this first code cell for normal runs. Remove `"X"` or `"H"`
from `TEST_GATES` to run a single probe. Set `QFT_DECODER_WIDTHS = ()` to run
only the single-qubit baseline.


In [ ]:
RUN_MODE = "ideal"  # "ideal" for local Aer or "hardware" for IBM Runtime
TEST_GATES = ("X", "H")
QFT_DECODER_WIDTHS = (2, 4)  # Choose 2, 4, or both. Use () to skip.

# Local Aer noise. These settings are ignored in hardware mode.
ENABLE_SIM_NOISE = True
SIM_GATE_ERROR_RATE = 0.01
SIM_READOUT_ERROR_RATE = 0.02

SHOTS = 4096
SEED = 20261028
AER_EXPORT_FILE = "results/gate_baseline_aer.json"
HARDWARE_EXPORT_FILE = "results/gate_baseline_hardware.json"
QFT_DECODER_EXPORT_DIR = "results"

# Hardware only. Running in hardware mode authenticates and submits a job.
API_KEY_FILE = "apikey.json"
QPU = "ibm_fez"
PHYSICAL_QUBIT = 0
QFT_PHYSICAL_QUBITS = (0, 1, 2, 3)  # The 2-qubit run uses the first pair.
HARDWARE_OPTIMIZATION_LEVEL = 1


## Single-qubit baseline

Each bar is the mean wrong-outcome percentage across `|0⟩`, `|1⟩`, `|+⟩`,
and `|−⟩`. The control and round trip use the same preparations and
measurement bases.

Run All also exports the pooled no-gate Z-control counts for prepared `0`
and `1`. They estimate effective errors from state preparation and readout.
Aer and hardware use separate files. The repeated-secret notebook reads the
hardware export by default when its baseline switch is enabled.


In [ ]:
import json
from datetime import datetime, timezone
from pathlib import Path

import matplotlib.pyplot as plt
from qiskit import QuantumCircuit, transpile

from quantum_study.qft_decoder import decoder_circuits

if RUN_MODE not in {"ideal", "hardware"}:
    raise ValueError("RUN_MODE must be 'ideal' or 'hardware'")
if not TEST_GATES or any(gate not in {"X", "H"} for gate in TEST_GATES):
    raise ValueError("TEST_GATES must contain X, H, or both")
if type(SHOTS) is not int or SHOTS < 1:
    raise ValueError("SHOTS must be a positive integer")

decoder_batches = {width: decoder_circuits(width) for width in QFT_DECODER_WIDTHS}
if RUN_MODE == "hardware" and decoder_batches:
    largest_width = max(decoder_batches)
    layout = QFT_PHYSICAL_QUBITS[:largest_width]
    if len(layout) != largest_width or len(set(layout)) != largest_width:
        raise ValueError("QFT_PHYSICAL_QUBITS must list distinct qubits for the largest decoder.")

if RUN_MODE == "ideal":
    from qiskit_aer import AerSimulator
    from qiskit_aer.noise import NoiseModel, ReadoutError, depolarizing_error

    noise_model = None
    if ENABLE_SIM_NOISE:
        if any(not 0 <= rate <= 1 for rate in (SIM_GATE_ERROR_RATE, SIM_READOUT_ERROR_RATE)):
            raise ValueError("Simulation error rates must be between 0 and 1")
        noise_model = NoiseModel()
        if SIM_GATE_ERROR_RATE:
            noise_model.add_all_qubit_quantum_error(
                depolarizing_error(SIM_GATE_ERROR_RATE, 1), ["x", "h"]
            )
        if SIM_READOUT_ERROR_RATE:
            flip = SIM_READOUT_ERROR_RATE
            noise_model.add_all_qubit_readout_error(
                ReadoutError([[1 - flip, flip], [flip, 1 - flip]])
            )
    backend = AerSimulator(noise_model=noise_model)
    has_noise = noise_model is not None and not noise_model.is_ideal()
    mode_label = "noisy local Aer" if has_noise else "noiseless local Aer"
else:
    from qiskit.transpiler import generate_preset_pass_manager
    from qiskit_ibm_runtime import QiskitRuntimeService
    from qiskit_ibm_runtime.executor_sampler import Sampler

    credentials = json.loads(Path(API_KEY_FILE).read_text(encoding="utf-8"))
    token = credentials.get("apikey")
    if not isinstance(token, str) or not token.strip():
        raise ValueError(f"{API_KEY_FILE} must contain a non-empty 'apikey' field")
    service_options = {"channel": "ibm_quantum_platform", "token": token.strip()}
    if credentials.get("crn"):
        service_options["instance"] = credentials["crn"]
    backend = QiskitRuntimeService(**service_options).backend(QPU)
    mode_label = f"{QPU} hardware"

states = (("Z", 0), ("Z", 1), ("X", 0), ("X", 1))
run_types = ("Control", "Gate round trip")
circuits, cases = [], []
for gate_name in TEST_GATES:
    for basis, bit in states:
        for run_type in run_types:
            circuit = QuantumCircuit(1, 1)
            if bit:
                circuit.x(0)
            if basis == "X":
                circuit.h(0)
            if run_type == "Gate round trip":
                apply_gate = getattr(circuit, gate_name.lower())
                circuit.barrier()
                apply_gate(0)
                circuit.barrier()
                apply_gate(0)
                circuit.barrier()
            if basis == "X":
                circuit.h(0)
            circuit.measure(0, 0)
            circuits.append(circuit)
            cases.append((gate_name, run_type, basis, bit))

if RUN_MODE == "ideal":
    submitted_circuits = circuits.copy()
    for batch, _ in decoder_batches.values():
        submitted_circuits.extend(transpile(batch, backend=backend, optimization_level=0))
    counts_by_circuit = backend.run(
        submitted_circuits, shots=SHOTS, seed_simulator=SEED
    ).result().get_counts()
else:
    groups = [(circuits, [PHYSICAL_QUBIT])] + [
        (batch, list(QFT_PHYSICAL_QUBITS[:width]))
        for width, (batch, _) in decoder_batches.items()
    ]
    submitted_circuits = []
    for batch, layout in groups:
        pass_manager = generate_preset_pass_manager(
            backend=backend,
            optimization_level=HARDWARE_OPTIMIZATION_LEVEL,
            initial_layout=layout,
            seed_transpiler=SEED,
        )
        submitted_circuits.extend(pass_manager.run(batch))
    job = Sampler(mode=backend).run(submitted_circuits, shots=SHOTS)
    job_id = job.job_id()
    print("IBM Runtime job ID:", job_id)
    counts_by_circuit = []
    for result in job.result():
        registers = tuple(result.data.keys())
        if len(registers) != 1:
            raise RuntimeError(f"Expected one measurement register, found {registers}")
        counts_by_circuit.append(getattr(result.data, registers[0]).get_counts())

if len(counts_by_circuit) != len(submitted_circuits):
    raise RuntimeError("Backend result count does not match the circuit count")
rates = {(gate, run): [] for gate in TEST_GATES for run in run_types}
assignment_counts = {0: {"shots": 0, "errors": 0}, 1: {"shots": 0, "errors": 0}}
for (gate, run, basis, expected), counts in zip(cases, counts_by_circuit[:len(cases)]):
    shots_observed = sum(counts.values())
    wrong = sum(
        count
        for outcome, count in counts.items()
        if int(outcome.replace(" ", ""), 2) != expected
    )
    rates[(gate, run)].append(100 * wrong / shots_observed)
    if run == "Control" and basis == "Z":
        assignment_counts[expected]["shots"] += shots_observed
        assignment_counts[expected]["errors"] += wrong
mean_rates = {key: sum(values) / len(values) for key, values in rates.items()}

backend_name = getattr(backend, "name", QPU if RUN_MODE == "hardware" else "aer_simulator")
if callable(backend_name):
    backend_name = backend_name()
source = "hardware" if RUN_MODE == "hardware" else "aer"
export = {
    "source": source,
    "backend": str(backend_name),
    "physical_qubit": int(PHYSICAL_QUBIT) if RUN_MODE == "hardware" else None,
    "shots_0": int(assignment_counts[0]["shots"]),
    "errors_0": int(assignment_counts[0]["errors"]),
    "shots_1": int(assignment_counts[1]["shots"]),
    "errors_1": int(assignment_counts[1]["errors"]),
    "saved_at": datetime.now(timezone.utc).isoformat().replace("+00:00", "Z"),
}
if RUN_MODE == "hardware":
    export["job_id"] = str(job_id)
export_path = Path(HARDWARE_EXPORT_FILE if RUN_MODE == "hardware" else AER_EXPORT_FILE)
export_path.parent.mkdir(parents=True, exist_ok=True)
export_path.write_text(json.dumps(export, indent=2) + "\n", encoding="utf-8")
print(f"Saved assignment-error counts to {export_path}")

positions = list(range(len(TEST_GATES)))
width = 0.36
fig, ax = plt.subplots(figsize=(7, 4))
control_bars = ax.bar(
    [x - width / 2 for x in positions],
    [mean_rates[(gate, "Control")] for gate in TEST_GATES],
    width,
    label="Control",
)
round_trip_bars = ax.bar(
    [x + width / 2 for x in positions],
    [mean_rates[(gate, "Gate round trip")] for gate in TEST_GATES],
    width,
    label="Gate round trip",
)
ax.set_title(f"BB84 {'/'.join(TEST_GATES)} round-trip baseline ({mode_label})")
ax.set_ylabel("Mean wrong outcomes (%)")
ax.set_xlabel("Tested logical gate")
ax.set_xticks(positions, TEST_GATES)
ax.legend()
ax.grid(axis="y", alpha=0.25)
for bars in (control_bars, round_trip_bars):
    ax.bar_label(bars, fmt="%.2f", padding=3, fontsize=8)
fig.tight_layout()
plt.show()


## Baseline interpretation and sources

Within each gate, compare **Gate round trip** with **Control**. Any excess is
an end-to-end circuit effect, not isolated single-gate infidelity or a QKD
security result. The synthetic Aer noise also affects X/H instructions used
for state preparation and measurement-basis changes. Hardware results also
include transpilation, routing, preparation, and readout effects.

Sources: [BB84 state encoding](https://arxiv.org/abs/2003.06557), Qiskit Aer
[`NoiseModel`](https://qiskit.github.io/qiskit-aer/apidocs/aer_noise.html),
and IBM's [Sampler and transpilation guide](https://quantum.cloud.ibm.com/docs/en/guides/sampler-options).


## Bob's inverse QFT on known words

For every input word at each selected block size, the control prepares that word in the computational basis and measures it. The decoder circuit prepares the corresponding Fourier state directly with Hadamards and single-qubit phase rotations, runs the inverse QFT, and measures the decoded word. **Bit 0** is the least significant bit, shown at the right of a word. The whole-word rate counts shots with at least one wrong bit.

The hardware results include preparation, native gate decomposition, routing, and readout. Local Aer uses the X/H and readout noise settings above and ideal two-qubit gates. The decoder export records outcome counts by expected word and the two-qubit operations used. The repeated-secret notebook uses the separate single-qubit Z-control export for its readout model.

[Two-pass decoder, §4.1, Eq. (3)](https://arxiv.org/abs/2404.12507v2) · [Qiskit QFT gate](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.circuit.library.QFTGate)


In [ ]:
if decoder_batches:
    import pandas as pd

    from quantum_study.qft_decoder import decoder_summary

    decoder_results = []
    offset = len(cases)
    for width, (_, word_cases) in decoder_batches.items():
        limit = offset + len(word_cases)
        counts = counts_by_circuit[offset:limit]
        compiled = submitted_circuits[offset:limit]
        native_two_qubit_gates = sorted({
            instruction.operation.name
            for circuit in compiled
            for instruction in circuit.data
            if instruction.operation.num_qubits == 2 and instruction.operation.name != "barrier"
        })
        summary = decoder_summary(width, word_cases, counts)
        columns = {
            "case": "Circuit",
            "shots": "Shots",
            "word_errors": "Wrong words",
            "word_error_rate": "Whole-word error",
            **{f"bit_{bit}_error_rate": f"Bit {bit} error" for bit in range(width)},
        }
        table = pd.DataFrame(summary)[list(columns)]
        for column in table.filter(like="_error_rate"):
            table[column] = table[column].map("{:.2%}".format)
        print(f"{width}-qubit inverse QFT ({mode_label})")
        display(table.rename(columns=columns))
        decoder_results.append({
            "block_qubits": width,
            "initial_layout": list(QFT_PHYSICAL_QUBITS[:width]) if source == "hardware" else None,
            "two_qubit_operations": native_two_qubit_gates,
            "summary": summary,
            "counts_by_word": [
                {
                    "case": case,
                    "expected_word": format(word, f"0{width}b"),
                    "outcomes": {outcome: int(count) for outcome, count in outcomes.items()},
                }
                for (case, word), outcomes in zip(word_cases, counts)
            ],
        })
        offset = limit

    decoder_export = {field: export[field] for field in ("source", "backend", "saved_at")}
    if source == "hardware":
        decoder_export["job_id"] = export["job_id"]
        decoder_export["optimization_level"] = HARDWARE_OPTIMIZATION_LEVEL
    else:
        decoder_export["x_h_noise"] = SIM_GATE_ERROR_RATE if ENABLE_SIM_NOISE else 0.0
        decoder_export["readout_noise"] = SIM_READOUT_ERROR_RATE if ENABLE_SIM_NOISE else 0.0
    decoder_export["seed"] = SEED
    decoder_export["results"] = decoder_results
    decoder_path = Path(QFT_DECODER_EXPORT_DIR) / f"qft_decoder_{source}.json"
    decoder_path.parent.mkdir(parents=True, exist_ok=True)
    decoder_path.write_text(json.dumps(decoder_export, indent=2) + "\n", encoding="utf-8")
    print(f"Saved decoder outcomes to {decoder_path}")
